# RAG Failure Analysis

This notebook saves the actual failed retrieval results, retrieved content, failure analyses, and a combined report for the short-query and long-query evaluation sets.

In [1]:
import json
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path("failure_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_json(data, filename):
    path = OUTPUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)
    print(f"Saved: {path}")

## 1. Helper functions

In [2]:
def get_rank(item):
    """Extract retrieved rank from common result formats."""
    if "rank" in item:
        return item["rank"]
    if "top1_rank" in item:
        return item["top1_rank"]
    if "retrieved_rank" in item:
        return item["retrieved_rank"]
    return None


def is_top1_failure(item):
    rank = get_rank(item)
    return rank is not None and rank != 1

## 2. Check input variables

The notebook expects `short_query_results` and `long_query_results` to already exist in the kernel.

In [3]:
if "short_query_results" not in globals():
    raise NameError(
        "short_query_results was not found. "
        "Load/generate the 117 short-query results first."
    )

if "long_query_results" not in globals():
    raise NameError(
        "long_query_results was not found. "
        "Load/generate the 41 long-query results first."
    )

print(f"Short-query results: {len(short_query_results)}")
print(f"Long-query results: {len(long_query_results)}")

NameError: short_query_results was not found. Load/generate the 117 short-query results first.

## 3. Extract failed short-query results

In [ ]:
short_failures = [
    item for item in short_query_results
    if isinstance(item, dict) and is_top1_failure(item)
]

save_json(short_failures, "short_query_failures.json")

print(f"Short-query failures: {len(short_failures)} / {len(short_query_results)}")

## 4. Extract failed long-query results

In [ ]:
long_failures = [
    item for item in long_query_results
    if isinstance(item, dict) and is_top1_failure(item)
]

save_json(long_failures, "long_query_failures.json")

print(f"Long-query failures: {len(long_failures)} / {len(long_query_results)}")

## 5. Failure categories

These categories are initialized here. They are only counted automatically if your result records already contain a `failure_category` field.

In [ ]:
failure_categories = [
    "semantic_mismatch",
    "wrong_section",
    "generic_query",
    "chunk_overlap",
    "similar_policy",
    "embedding_limitation",
    "metadata_issue",
    "other"
]

short_category_counts = {category: 0 for category in failure_categories}
long_category_counts = {category: 0 for category in failure_categories}

for item in short_failures:
    category = item.get("failure_category")
    if category in short_category_counts:
        short_category_counts[category] += 1

for item in long_failures:
    category = item.get("failure_category")
    if category in long_category_counts:
        long_category_counts[category] += 1

print("Short categories:", short_category_counts)
print("Long categories:", long_category_counts)

## 6. Save short-query analysis

In [ ]:
short_query_analysis = {
    "dataset": "short_queries",
    "total_queries": len(short_query_results),
    "failed_top1": len(short_failures),
    "failure_rate": (
        len(short_failures) / len(short_query_results)
        if len(short_query_results) > 0 else 0
    ),
    "failure_categories": short_category_counts,
    "failures": short_failures
}

save_json(short_query_analysis, "short_query_analysis.json")

## 7. Save long-query analysis

In [ ]:
long_query_analysis = {
    "dataset": "long_queries",
    "total_queries": len(long_query_results),
    "failed_top1": len(long_failures),
    "failure_rate": (
        len(long_failures) / len(long_query_results)
        if len(long_query_results) > 0 else 0
    ),
    "failure_categories": long_category_counts,
    "failures": long_failures
}

save_json(long_query_analysis, "long_query_analysis.json")

## 8. Baseline metrics

In [ ]:
baseline = {
    "short_queries": {
        "total": 117,
        "top1_accuracy": 75.21,
        "top3_recall": 86.32,
        "top5_recall": 87.18
    },
    "long_queries": {
        "total": 41,
        "top1_accuracy": 87.80,
        "top3_recall": 95.12,
        "top5_recall": 95.12
    }
}

baseline

## 9. Create combined failure-analysis report

In [ ]:
final_report = {
    "generated_at": datetime.now().isoformat(),
    "description": (
        "Failure analysis of the baseline RAG retrieval system. "
        "The report preserves the actual failed retrieval records "
        "for subsequent inspection."
    ),
    "baseline": baseline,
    "failure_counts": {
        "short_queries": len(short_failures),
        "long_queries": len(long_failures),
        "total_failures": len(short_failures) + len(long_failures)
    },
    "failure_rates": {
        "short_queries": (
            len(short_failures) / len(short_query_results)
            if len(short_query_results) > 0 else 0
        ),
        "long_queries": (
            len(long_failures) / len(long_query_results)
            if len(long_query_results) > 0 else 0
        )
    },
    "short_query_failure_categories": short_category_counts,
    "long_query_failure_categories": long_category_counts,
    "short_query_failures": short_failures,
    "long_query_failures": long_failures
}

save_json(final_report, "failure_analysis_report.json")

## 10. Final summary

In [ ]:
print("=" * 60)
print("FAILURE ANALYSIS COMPLETE")
print("=" * 60)

print(f"Output directory: {OUTPUT_DIR.resolve()}")

print("\nFiles created:")
print("1. short_query_failures.json")
print("2. long_query_failures.json")
print("3. short_query_analysis.json")
print("4. long_query_analysis.json")
print("5. failure_analysis_report.json")

print("\nSummary:")
print(f"Short queries: {len(short_failures)} failures / {len(short_query_results)} total")
print(f"Long queries: {len(long_failures)} failures / {len(long_query_results)} total")
print(f"Total failures: {len(short_failures) + len(long_failures)}")